<a href="https://colab.research.google.com/github/parsa-asgari/bahar-bootcamp/blob/main/bahar_RAG_LLM_Markdown.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Practice: RAG + LLM + Markdown

## 1. Install python packages

In [5]:
!pip install pinecone openai python-docx
!pip install transformers
!pip install  langchain-text-splitters
# !pip install langchain-pinecone langchain-openai

## 2. API Keys

In [2]:
# API KEYS
PINECONE_API_KEY=''
PINECONE_INDEX_NAME=''
OPENROUTER_API_KEY=''
GROQ_API_KEY=''
HUGGINGFACE_TOKEN=''

## 3. Prepare bahar_danesh_internal_policy.md markdown file

#### 1. Go to https://app.dawin.io/ and paste all the bahar_danesh_internal_policy.docx into it.
#### 2. Copy the markdown result from app.dawin.io and save it into a .md file.
#### 3. upload the file in colab

In [8]:

import os
import hashlib
from pathlib import Path

from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)



# Configuration
MARKDOWN_PATH = Path("bahar_danesh_internal_policy.md")

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150


def load_and_split_markdown(path: Path):
    # Read Markdown file
    markdown_text = path.read_text(encoding="utf-8")

    # First: split by Markdown headers
    headers_to_split_on = [
        ("#", "header_1"),
        ("##", "header_2"),
        ("###", "header_3"),
    ]

    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on,
        strip_headers=False,
    )

    header_documents = header_splitter.split_text(markdown_text)

    # Second: split large sections into smaller chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks = text_splitter.split_documents(header_documents)

    # Attach source metadata and create deterministic IDs
    for chunk in chunks:
        chunk.metadata["source"] = path.name

    ids = [
        hashlib.sha256(
            f"{path.name}:{i}:{chunk.page_content}".encode("utf-8")
        ).hexdigest()
        for i, chunk in enumerate(chunks)
    ]

    return chunks, ids



In [10]:
chunks, ids = load_and_split_markdown(MARKDOWN_PATH)

In [11]:
chunks[:2]

[Document(metadata={'source': 'bahar_danesh_internal_policy.md'}, page_content='<span style="line-height: 115%;">شرکت بهار دانش</span>  \n<span style="line-height: 115%;">معاونت اداری و سرمایه انسانی</span>  \n<span>سند شماره: BK-HR-1405-017</span>  \n<span style="line-height: 115%;">عنوان: آییننامه داخلی کارکنان شرکت بهار دانش</span>  \nتاریخ ابلاغ: ۱۴۰۵/۰۷/۱۵  \nنسخه: ۱.۰  \nسطح دسترسی: داخلی ـ آموزشی  \nوضعیت سند: معتبر در چارچوب سناریوی آموزشی  \nموضوع: تشریح ساعات کاری، مرخصی، دورکاری، آموزش، مزایا و فرایندهای اداری کارکنان'),
 Document(metadata={'header_1': '۱. هدف و دامنه کاربرد', 'source': 'bahar_danesh_internal_policy.md'}, page_content='# ۱. هدف و دامنه کاربرد  \nاین آییننامه با هدف ایجاد رویهای یکسان برای امور اداری و منابع انسانی شرکت خیالی بهار دانش تدوین شده است. مفاد آن در سناریوی آموزشی حاضر برای تمام کارکنان تماموقت و پارهوقت شرکت، مگر در مواردی که صراحتاً استثنا شده باشد، قابل اعمال است.  \nتبصره: این سند ساختگی است و نباید بهعنوان مقررات واقعی یک شرکت یا مشاوره حقوقی

In [18]:
chunks[0].page_content

'<span style="line-height: 115%;">شرکت بهار دانش</span>  \n<span style="line-height: 115%;">معاونت اداری و سرمایه انسانی</span>  \n<span>سند شماره: BK-HR-1405-017</span>  \n<span style="line-height: 115%;">عنوان: آییننامه داخلی کارکنان شرکت بهار دانش</span>  \nتاریخ ابلاغ: ۱۴۰۵/۰۷/۱۵  \nنسخه: ۱.۰  \nسطح دسترسی: داخلی ـ آموزشی  \nوضعیت سند: معتبر در چارچوب سناریوی آموزشی  \nموضوع: تشریح ساعات کاری، مرخصی، دورکاری، آموزش، مزایا و فرایندهای اداری کارکنان'

In [16]:
chunks[0].metadata['source']

'bahar_danesh_internal_policy.md'

## 4. Insert to Pinecone

In [12]:
import os, time
from pinecone import Pinecone, SchemaBuilder

pc = Pinecone(api_key=PINECONE_API_KEY)

schema = (
    SchemaBuilder()
      .add_dense_vector_field(name="embedding", dimension=1024, metric="cosine")
      .build()
)
if not pc.indexes.exists(name=PINECONE_INDEX_NAME):
    pc.indexes.create(name=PINECONE_INDEX_NAME, schema=schema)

while not pc.indexes.describe(name=PINECONE_INDEX_NAME).status.ready:
    time.sleep(2)

index = pc.Index(name=PINECONE_INDEX_NAME)

### 4.1 Setup Embedding Model

In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('heydariAI/persian-embeddings')



modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.97k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

In [19]:
import os
from pathlib import Path
from pinecone import Pinecone

# Pinecone client
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)
chunks_inserted = 0
for i,chunk in enumerate(chunks):
    embedding = model.encode(chunk.page_content)
    document = [{
            "_id": f"{i}",
            "embedding": embedding.tolist(),
            "text": chunk.page_content,
            "source": chunk.metadata['source'],
        }]
    result = index.documents.upsert(
        namespace="__default__",
        documents=document,
    )
    chunks_inserted = chunks_inserted + 1

print(
    f"Uploaded {chunks_inserted} chunks"
)
print("Finished uploading documents.")


Uploaded 15 chunks
Finished uploading documents.


# 5. RAG + LLM

In [22]:
from openai import OpenAI
from pinecone import DenseVectorQuery
# LLM
llm_client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

MODEL = "openai/gpt-oss-20b"

## Doing a real RAG

PROMPT = '''
*INSTRUCTION*
You are a helpful assistant representing 'بهار دانش', a telecom company in tehran, iran.
You are deployed to answer user questions.
Greet the user, if you have no conversation history.
Be polite, be gentle.
use the context to answer user questions as best as you can.
Don't answer questions unrelated to 'بهار دانش'.

*CONTEXT*
{context}
'''

query = "در مورد ساعات کاری شرکت به من بگو"

model = SentenceTransformer('heydariAI/persian-embeddings')
sentence_emb = model.encode(query)
results = index.documents.search(
    namespace="__default__",
    top_k=10,
    score_by=[DenseVectorQuery(field="embedding", values=sentence_emb)],
    include_fields=["text", "source"],
)

context = [doc.text for doc in results.matches]


response = llm_client.responses.create(
    model=MODEL,
    instructions=PROMPT.format(context=context),
    input="در مورد ساعات کاری شرکت به من بگو",
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [23]:
print(response.output[1].content[0].text)

سلام!  
ساعات کاری عادی شرکت بهار دانش به‌صورت زیر است:  

- **روزهای کاری:** شنبه تا چهارشنبه  
- **ساعت آغاز و پایان:** ۸:۳۰ صبح تا ۱۷:۰۰ عصر  
- **زمان استراحت روزانه:** ۱۲:۳۰ ظهر تا ۱۳:۱۵ بعد از ظهر (این زمان به‌عنوان زمان کاری محسوب نمی‌شود)  
- **ثبت حضور و غیاب:** کارکنان باید ورود و خروج خود را از طریق سامانه حضور و غیاب «بهارکار» ثبت کنند.  

همچنین، در صورت تأخیر بیش از ۱۵ دقیقه، باید همان روز به مدیر مستقیم اطلاع داده شود؛ و در صورتی که در یک ماه سه بار تأخیر ثبت شود، نیاز به گفتگوی پیگیری با مدیر مستقیم و واحد سرمایه انسانی است.  

اگر سؤال دیگری در مورد ساعت‌های کاری یا روند ثبت حضور دارید، خوشحال می‌شوم کمک کنم!
